# Joint lower bound versus sequential budgets

Comparison for paper §3.5.


## Saved-result reuse

`REUSE_SAVED_RESULTS=True` reuses compatible sequential DP results. Greedy and the joint integer program are evaluated for each selected case.


## Budgets

Compare the joint integer-program lower bound with sequential DP and Greedy budgets. The lower bound alone does not certify an achievable joint input state; see §3.5.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "problem-1-zero-error"
    if (working_directory / "problem-1-zero-error").is_dir()
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))
for module in ("run_config", "sensor_budget_analysis", "global_minimum", "phase_data", "optimal", "eval"):
    sys.modules.pop(module, None)
from sensor_budget_analysis import (
    run_budget_analysis, plot_budget_comparison, configure_style, save_figure,
)
print("Project directory:", PROJECT_DIRECTORY)


## Configuration

Choose `RUN_MODE="test"` or `"full"`; final figures use full results.


In [ ]:
RUN_MODE = "full"  # "test" or "full"
REUSE_SAVED_RESULTS = True  # False forces a fresh DP solve for every selected case.

if RUN_MODE not in ("test", "full"):
    raise ValueError('RUN_MODE must be "test" or "full"')
print("Run mode:", RUN_MODE)
print("Expected standard benchmark cases:", 132 if RUN_MODE == "test" else 39_600)


## Compute budgets


In [ ]:
analysis = run_budget_analysis(
    PROJECT_DIRECTORY, include_global_minimum=True, run_mode=RUN_MODE,
    reuse_saved=REUSE_SAVED_RESULTS,
)
rows = analysis["rows"]
summary = analysis["summary"]
OUTPUT_DIRECTORY = analysis["output_directory"]
assert all(row["N_min_global"] <= row["N_min"] <= row["N_greedy"] for row in rows)
print(f"Verified all {len(rows):,} paired instances.")
print("Output:", OUTPUT_DIRECTORY)

## Matched budget summary


In [ ]:
def show_summary(rows, include_global=False):
    header = r"| Dimension | K | Cases | Mean $N_{\min}^{\mathrm{seq}}$ |"
    separator = "|---|---:|---:|---:|"
    if include_global:
        header += r" Mean $N_{\mathrm{greedy}}^{\mathrm{seq}}$ | Mean $N_{\min}^{\mathrm{global}}$ | Mean gap | Equal budgets (%) |"
        separator += "---:|---:|---:|---:|"
    lines = [header, separator]
    for row in rows:
        line = (f"| {row['dimension']} | {row['K']} | {row['case_count']} | "
                f"{row['mean_N_min_opt']:.3f} |")
        if include_global:
            line += (f" {row['mean_N_greedy']:.3f} | {row['mean_N_min_global']:.3f} | {row['mean_sensor_gap']:.3f} | "
                     f"{100 * row['equal_budget_fraction']:.1f} |")
        lines.append(line)
    display(Markdown("\n".join(lines)))

show_summary(summary, include_global=True)

## Budget comparison


In [ ]:
figure = plot_budget_comparison(summary)
if RUN_MODE == "test":
    figure.suptitle(f"Test run: {len(rows):,} cases (subset of benchmark)")
    figure.tight_layout(rect=(0, 0, 1, 0.94))
save_figure(figure, OUTPUT_DIRECTORY / "figures", "global_vs_seq_by_k")
display(figure)
plt.close(figure)

## Outputs

Per-case budgets, summaries and figures: `results/global/{test,full}/`.
